In [1]:
%pip install yfinance scikit-learn hmmlearn matplotlib

  Using cached yfinance-1.5.1-py2.py3-none-any.whl.metadata (6.2 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached peewee-4.2.2-py3-none-any.whl.metadata (10 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
Using cached yfinance-1.5.1-py2.py3-none-any.whl (144 kB)
Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl (1.7 MB)
Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)
Using cached peewee-4.2.2-py3-none-any.whl (172 kB)

   ---------------------------------------- 0/7 [peewee]
  Attempting uninstall: cffi
   ---------------------------------------- 0/7 [peewee]
   ----------------- ---------------------- 3/7 [cffi]
    Found existing installation: cffi 1.17.1
   ----------------- ---------------------- 3/7 [cffi]
    Uninstalling cffi-1.17.1:
   ----------------- ---------------------- 3/7 [cffi]
   ----------------- ---------------------- 3/7 [cffi]
   ----------------- ---------------------- 3/7 [cffi]


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

import matplotlib.pyplot as plt

In [3]:

data = yf.download(
    "^NSEI",
    start="2010-01-01",
    end="2026-01-01",
    auto_adjust=True
)
data.head()
print(data.columns)
print(data.shape)

[*********************100%***********************]  1 of 1 completed

MultiIndex([( 'Close', '^NSEI'),
            (  'High', '^NSEI'),
            (   'Low', '^NSEI'),
            (  'Open', '^NSEI'),
            ('Volume', '^NSEI')],
           names=['Price', 'Ticker'])
(3929, 5)


In [4]:
# Flatten MultiIndex columns
data.columns = data.columns.get_level_values(0)
# Convert Date index to normal column
data = data.reset_index()
# Remove column index name
data.columns.name = None
# Sort by date
data = data.sort_values("Date").reset_index(drop=True)
# Save clean dataset
data.to_csv("clean_data.csv", index=False)
print(data.columns)
print(data.shape)

data.head()

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
(3929, 6)


,Date,Close,High,Low,Open,Volume
0,2010-01-04,5232.200195,5238.450195,5167.100098,5200.899902,0
1,2010-01-05,5277.899902,5288.350098,5242.399902,5277.149902,0
2,2010-01-06,5281.799805,5310.850098,5260.049805,5278.149902,0
3,2010-01-07,5263.100098,5302.549805,5244.750000,5281.799805,0
4,2010-01-08,5244.750000,5276.750000,5234.700195,5264.250000,0


In [7]:

# Calculate log returns
data["Log_Return"] = np.log(
    data["Close"] / data["Close"].shift(1)
)
# garman-klass volatility
data["gk_volatility"] = np.sqrt(0.5*(np.log(data["High"]/data["Low"]))**2
    -(2*np.log(2)-1) * (np.log(data["Close"]/data["Open"]))**2)
# Range Feature
data["Range"] = (
    data["High"] - data["Low"]
) / data["Close"]
# Remove NaN values created by rolling window
data = data.dropna().reset_index(drop=True)


In [6]:
features = data[[
    "Date",
    "Log_Return",
    "gk_volatility",
    "Range"
]].copy()
features.to_csv("features.csv", index=False)
print(features.head())
print(features.shape)

        Date  Log_Return  gk_volatility     Range
0 2010-01-05    0.008696       0.006170  0.008706
1 2010-01-06    0.000739       0.006783  0.009618
2 2010-01-07   -0.003547       0.007430  0.010982
3 2010-01-08   -0.003493       0.005166  0.008018
4 2010-01-11    0.000886       0.007806  0.011316
(3928, 4)


In [9]:

X = features[["Log_Return", "gk_volatility","Range"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled[:5])
print(X_scaled.shape)


[[ 0.79411083 -0.18534108 -0.39058234]
 [ 0.03154703 -0.05568995 -0.27529471]
 [-0.37910059  0.08131918 -0.10282629]
 [-0.37392238 -0.39794605 -0.47765102]
 [ 0.04568628  0.16083911 -0.06065157]]
(3928, 3)


In [17]:

hmm_model = GaussianHMM(
    n_components=5,
    covariance_type="full",
    n_iter=100,
    random_state=42
)

hmm_model.fit(X_scaled)
log_likelihood = hmm_model.score(X_scaled)
print("Training completed")
print("Number of states:", hmm_model.n_components)
print("Log Likelihood:", log_likelihood)


Training completed
Number of states: 5
Log Likelihood: -6192.945288476841


In [18]:


joblib.dump(
    hmm_model,
    "hmm_model.pkl"
)

['hmm_model.pkl']

In [19]:
print("State Means:")
print(hmm_model.means_)
print("\nTransition Matrix:")
print(hmm_model.transmat_)


State Means:
[[ 8.57490567e-01 -2.49552636e-01 -1.19207238e-01]
 [ 7.49420621e-02 -2.70930633e-01 -4.18003928e-01]
 [-6.93670510e-01 -1.15261554e-01 -1.04645216e-02]
 [ 1.41301970e+00  2.29370864e+01  1.88946840e+01]
 [-2.85710757e-01  1.32970086e+00  1.44865062e+00]]

Transition Matrix:
[[2.34551697e-001 5.39062478e-001 2.07011880e-001 1.30154319e-003
  1.80724017e-002]
 [2.15746504e-001 4.60935145e-001 2.64421491e-001 3.42835694e-134
  5.88968605e-002]
 [2.07788142e-001 4.25563456e-001 3.03655373e-001 1.07782332e-177
  6.29930290e-002]
 [0.00000000e+000 0.00000000e+000 4.81389860e-001 0.00000000e+000
  5.18610140e-001]
 [5.15037146e-002 2.21830160e-001 4.73369786e-002 1.87177172e-003
  6.77457375e-001]]


In [21]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [23]:
features.to_csv(
    "/content/drive/MyDrive/features.csv",
    index=False
)

print("features.csv saved")

OSError: Cannot save file into a non-existent directory: '\content\drive\MyDrive'

In [24]:
states = hmm_model.predict(X_scaled)

features["State"] = states

print("\nState Counts:")
print(features["State"].value_counts())

print("\nState Statistics:")
print(
    features.groupby("State")[
        ["Log_Return", "gk_volatility", "Range"]
    ].mean()
)

states_df = features[
    ["Date", "State"]
].copy()


State Counts:
State
1    1902
2     830
0     695
4     499
3       2
Name: count, dtype: int64

State Statistics:
       Log_Return  gk_volatility     Range
State                                     
0        0.010304       0.006082  0.011470
1        0.001151       0.005710  0.008406
2       -0.007547       0.006637  0.012174
3        0.015155       0.115402  0.161238
4       -0.003022       0.013724  0.023939


In [25]:
states_df.to_csv("states.csv", index=False)

In [ ]:
states_df.to_csv(
    "/content/drive/MyDrive/states.csv",
    index=False
)
print("states.csv saved successfully")

states.csv saved successfully


In [ ]:
state_probs = hmm_model.predict_proba(
    X_scaled
)
prob_df = pd.DataFrame(
    state_probs,
    columns=[
        f"State_{i}"
        for i in range(
            hmm_model.n_components
        )
    ]
)
prob_df["Date"] = features["Date"]
prob_df.to_csv(
    "state_probabilities.csv",
    index=False
)
prob_df.to_csv(
    "/content/drive/MyDrive/state_probabilities.csv",
    index=False
)
print("state_probabilities.csv saved")

state_probabilities.csv saved


In [ ]:
transition_df = pd.DataFrame(
    hmm_model.transmat_
)

transition_df.to_csv(
    "transition_matrix.csv",
    index=False
)
transition_df.to_csv(
    "/content/drive/MyDrive/transition_matrix.csv",
    index=False
)
print("transition_matrix.csv saved")

transition_matrix.csv saved


In [28]:
import numpy as np

stats = []

for state in range(
    hmm_model.n_components
):

    state_data = features[
        features["State"] == state
    ]

    mean_return = (
        state_data["Log_Return"]
        .mean()
    )

    gk_volatility = (
        np.sqrt(
    0.5*(np.log(data["High"]/data["Low"]))**2
    -(2*np.log(2)-1) * (np.log(data["Close"]/data["Open"]))**2)
    )

    sharpe = (
        mean_return /
        gk_volatility
        if gk_volatility > 0
        else np.nan
    )

    downside = state_data[
        state_data["Log_Return"] < 0
    ]["Log_Return"]

    sortino = (
        mean_return /
        downside.std()
        if len(downside) > 1
        else np.nan
    )

    stats.append({
        "State": state,
        "Frequency": len(state_data),
        "Mean_Return": mean_return,
        "gk_Volatility": gk_volatility,
        "Sharpe_Ratio": sharpe,
        "Sortino_Ratio": sortino
    })

state_characteristics = pd.DataFrame(
    stats
)

state_characteristics.to_csv(
    "state_characteristics.csv",
    index=False
)
state_characteristics.to_csv(
    "/content/drive/MyDrive/state_characteristics.csv",
    index=False
)
state_characteristics

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [33]:
persistence = []

for i in range(
    hmm_model.n_components
):

    pii = hmm_model.transmat_[i, i]

    persistence.append({
        "State": i,
        "Persistence": pii,
        "Expected_Duration":
            1/(1-pii)
    })

persistence_df = pd.DataFrame(
    persistence
)

persistence_df.to_csv(
    "persistence_metrics.csv",
    index=False
)
persistence_df.to_csv(
    "/content/drive/MyDrive/persistence_metrics.csv",
    index=False
)
persistence_df

OSError: Cannot save file into a non-existent directory: '\content\drive\MyDrive'

In [34]:
import joblib
joblib.dump(
    hmm_model,
    "hmm_model.pkl"
)
joblib.dump(
    scaler,
    "scaler.pkl"
)
print("Model and scaler saved")

Model and scaler saved


In [35]:
plot_data = data.copy()

plot_data = plot_data.iloc[-len(states):].copy()

plot_data["State"] = states

plot_data.head()

ValueError: Length of values (3928) does not match length of index (3927)

In [36]:


plt.figure(figsize=(15,6))

for state in sorted(plot_data["State"].unique()):
    mask = plot_data["State"] == state

    plt.scatter(
        plot_data.loc[mask, "Date"],
        plot_data.loc[mask, "Close"],
        s=5,
        label=f"State {state}"
    )

plt.plot(
    plot_data["Date"],
    plot_data["Close"],
    alpha=0.3
)

plt.title("NIFTY 50 Regimes")
plt.xlabel("Date")
plt.ylabel("Close")
plt.legend()
plt.show()
plt.savefig(
    "/content/drive/MyDrive/plot_4state_range.png",
    dpi=300,
    bbox_inches="tight"
)

KeyError: 'State'

<Figure size 1500x600 with 0 Axes>

In [ ]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'drive', 'features.csv', 'clean_data.csv', 'state_probabilities.csv', 'transition_matrix.csv', 'state_characteristics.csv', 'states.csv', 'scaler.pkl', 'hmm_model.pkl', 'persistence_metrics.csv', 'sample_data']
